# Test słowników

In [1]:
import pandas as pd
import json

with open("./keywords/positive_keywords.json", "r", encoding="utf-8") as f:
    positive_json = json.load(f)

with open("./keywords/negative_keywords.json", "r", encoding="utf-8") as f:
    negative_json = json.load(f)


print(f"Loaded {len(positive_json)} positive keywords.")
print(f"Loaded {len(negative_json)} negative keywords.")

Loaded 2699 positive keywords.
Loaded 2420 negative keywords.


In [2]:
import ahocorasick

A = ahocorasick.Automaton()

for phrase, value in positive_json.items():
    A.add_word(phrase.lower(), ("pos", phrase, value))

for phrase, value in negative_json.items():
    A.add_word(phrase.lower(), ("neg", phrase, value))

A.make_automaton()

In [3]:
df_maps = pd.read_csv("../data/sample_10k_maps_with_url_status.csv")

In [4]:
def compute_score_full_words(text):
    if not isinstance(text, str):
        text = str(text)

    text_lower = text.lower()
    score = 0
    used_phrases = set()
    pos_keys = []
    neg_keys = []

    for end_index, (ptype, phrase, value) in A.iter(text_lower):
        start_index = end_index - len(phrase) + 1

        if (start_index == 0 or not text_lower[start_index-1].isalnum()) and \
           (end_index == len(text_lower)-1 or not text_lower[end_index+1].isalnum()):

            if phrase not in used_phrases:
                score += value
                used_phrases.add(phrase)
                if ptype == "pos":
                    pos_keys.append(phrase)
                else:
                    neg_keys.append(phrase)

    return score, pos_keys, neg_keys

In [5]:
df_maps[["score", "pos", "neg"]] = df_maps["text"].apply(
    lambda x: pd.Series(compute_score_full_words(x))
)

df_maps[["text", "url", "url_status", "score", "pos", "neg"]]

,text,url,url_status,score,pos,neg
0,Australia Nuvola,http://a0.fast-meteo.com/maps/static/t_Austral...,ERR,0.0,[],[]
1,French Southern Territories Rubber Stamps vect...,https://cdn1.vectorstock.com/i/thumbs/22/75/fr...,OK,-5.0,[],"[stamps, image]"
2,New York - Látnivalók,http://7.hikb.at:80/img/iconmenu/attraction.png,OK,0.0,[],[]
3,Karte mit 5 bettel,https://aa1cc5ce659c48a883597af217799a86.objec...,ERR,0.0,[],[]
4,2004 Weekday Arterial Hourly Volumes: 6 p.m. t...,http://images.slideplayer.com/9/2524694/slides...,OK,0.0,[],[]
...,...,...,...,...,...,...
9997,Free Floorplan Designer floor plans and site p...,http://tse1.mm.bing.net/th?id=OIP.TUr6q0MyrZZZ...,OK,-11.0,[],"[floorplan, floor plans, plans]"
9998,Wolrd Map Coron Backpacker Guesthouse,https://static1.squarespace.com/static/52c24d1...,OK,0.0,[],[]
9999,c7d77b50192 Autopesula hinnakiri,https://www.tehnokuller.ee/images/tehnokuller-...,OK,0.0,[],[]
10000,kaart · Kentucky · Rood · patroon · amerika · ...,https://img.stockfresh.com/files/k/kyryloff/x/...,ERR,-8.0,[],"[patroon, stockfoto]"


In [6]:
only_valid_urls_df = df_maps[df_maps['url_status'] == 'OK'].reset_index(drop=True)

df_maps_positive = only_valid_urls_df[only_valid_urls_df['score'] > 0].reset_index(drop=True)
df_maps_negative = only_valid_urls_df[only_valid_urls_df['score'] < 0].reset_index(drop=True)
df_maps_neutral = only_valid_urls_df[only_valid_urls_df['score'] == 0].reset_index(drop=True)

print(f"Total valid URLs: {len(only_valid_urls_df)}")
print(f"Positive score maps: {len(df_maps_positive)}")
print(f"Negative score maps: {len(df_maps_negative)}")
print(f"Neutral score maps: {len(df_maps_neutral)}")

Total valid URLs: 5475
Positive score maps: 368
Negative score maps: 1901
Neutral score maps: 3206


In [7]:
import ipywidgets as widgets
from IPython.display import display, HTML
import numpy as np

class MapViewerBatch:
    def __init__(self, df, sort_by_score=None, batch_size=4, show_text=True):
        self.df = df.copy().reset_index(drop=True)
        self.batch_size = batch_size
        self.show_text = show_text

        if sort_by_score == 'asc':
            self.df.sort_values('score', ascending=True, inplace=True)
        elif sort_by_score == 'desc':
            self.df.sort_values('score', ascending=False, inplace=True)

        self.indices = np.arange(len(self.df))
        self.current_position = 0

        self.image_widget = widgets.Output()
        self.prev_button_top = widgets.Button(description='← Previous')
        self.next_button_top = widgets.Button(description='Next →')
        self.prev_button_bottom = widgets.Button(description='← Previous')
        self.next_button_bottom = widgets.Button(description='Next →')
        self.counter_widget = widgets.HTML()

        for btn in [self.prev_button_top, self.prev_button_bottom]:
            btn.on_click(self.show_previous)
        for btn in [self.next_button_top, self.next_button_bottom]:
            btn.on_click(self.show_next)

        self.nav_top = widgets.HBox([self.prev_button_top, self.counter_widget, self.next_button_top])
        self.nav_bottom = widgets.HBox([self.prev_button_bottom, self.next_button_bottom])

        self.container = widgets.VBox([self.nav_top, self.image_widget, self.nav_bottom])

    def get_current_indices(self):
        start = self.current_position
        end = min(self.current_position + self.batch_size, len(self.df))
        return self.indices[start:end]

    def show_images(self, b=None):
        self.image_widget.clear_output(wait=True)

        with self.image_widget:
            for idx in self.get_current_indices():
                row = self.df.iloc[idx]

                if not self.show_text:
                    info_html = f"""
                <div style='display:flex; align-items:flex-start; padding:5px; margin-bottom:15px; border:1px solid #eee; border-radius:5px;'>
                    <div style='flex:1; padding-right:10px;'>
                        <b>Score:</b> {row['score']}<br>
                        <b>Positive keywords:</b> {', '.join(row['pos']) if row['pos'] else 'None'}<br>
                        <b>Negative keywords:</b> {', '.join(row['neg']) if row['neg'] else 'None'}<br>
                    </div>
                    <div>
                        <img src="{row["url"]}" style="max-width:300px; border:2px solid #ddd; border-radius:5px;">
                    </div>
                </div>
                """
                else:
                    info_html = f"""
                    <div style='display:flex; align-items:flex-start; padding:5px; margin-bottom:15px; border:1px solid #eee; border-radius:5px;'>
                        <div style='flex:1; padding-right:10px;'>
                            <b>Text:</b> {row['text']}<br>
                            <b>Score:</b> {row['score']}<br>
                            <b>Positive keywords:</b> {', '.join(row['pos']) if row['pos'] else 'None'}<br>
                            <b>Negative keywords:</b> {', '.join(row['neg']) if row['neg'] else 'None'}<br>
                        </div>
                        <div>
                            <img src="{row["url"]}" style="max-width:300px; border:2px solid #ddd; border-radius:5px;">
                        </div>
                    </div>
                    """
                display(HTML(info_html))


            self.counter_widget.value = f"<center><b>Mapy {self.current_position + 1} – {min(self.current_position + self.batch_size, len(self.df))} / {len(self.df)}</b></center>"


    def show_next(self, b=None):
        if self.current_position + self.batch_size < len(self.indices):
            self.current_position += self.batch_size
            self.show_images()

    def show_previous(self, b=None):
        if self.current_position - self.batch_size >= 0:
            self.current_position -= self.batch_size
            self.show_images()

    def display(self):
        self.show_images()
        display(self.container)


In [9]:
MapViewerBatch(df_maps_positive, sort_by_score='desc', batch_size=10, show_text=True).display()